# 03 — Severity Model (Training + Demo)

## Training
- Trains the severity model and writes artifacts to `artifacts/<run_id>/`.
- Key files created in the new timestamp folder:
  - `severity_model.joblib` (the trained model)
  - `preprocessors.joblib` (the feature transformers used by the model)
  - `feature_cols.json` (feature columns expected at prediction time)
  - `severity_metrics.json` (training report/metrics)

### Optional GPU (Colab)
- Runtime → Change runtime type → GPU
- In config set:
  - `training.severity_model: xgboost`
  - `training.device: gpu`

## Demo / Inference (after training)
You can predict by passing **one JSON object** of feature values.
- Keys should match feature column names (all columns except excluded + targets).
- Missing keys are allowed (they become null/NaN and get imputed).
- Extra keys are ignored (you can include `_comment` fields).

In [9]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/retrofit

PROJECT_ROOT = '/content/drive/MyDrive/retrofit'
CONFIG = 'configs/colab_large.yaml'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/retrofit


In [10]:
!pip -q install -r requirements-colab.txt
!pip -q install -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bridge-retrofit (pyproject.toml) ... done


In [11]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', CONFIG,
    '--project-root', PROJECT_ROOT,
    'train',
    '--task', 'severity',
])

0

In [18]:
# --- Demo: load latest Severity run and predict ---
from pathlib import Path
import json
import joblib
import pandas as pd
from bridge_retrofit.preprocessing.pipeline import make_single_row_frame

artifacts_root = Path(PROJECT_ROOT) / "artifacts"
severity_runs = sorted(
    [
        d
        for d in artifacts_root.iterdir()
        if d.is_dir() and (d / "severity_model.joblib").exists() and (d / "preprocessors.joblib").exists()
    ],
    reverse=True,
 )
if not severity_runs:
    raise FileNotFoundError("No severity runs found. Run the Training cell first.")
run_dir = severity_runs[0]
print("Using severity run:", run_dir.name)

feature_cols = json.loads((run_dir / "feature_cols.json").read_text(encoding="utf-8"))
preprocessors = joblib.load(run_dir / "preprocessors.joblib")
severity_model = joblib.load(run_dir / "severity_model.joblib")

# Example JSON-like payload (extra keys like _comment are ignored by the pipeline)
payload = {
    "_comment": "Missing fields become NaN (same behavior as full pipeline/CLI).",
    "Bridge_Type": "Beam",
    "Material": "Steel",
    "Age_Years": 25,
    "Span_Length_m": 30,
    "Number_of_Spans": 3,
    "Failure_Type": "Scour",
}

# IMPORTANT: build the row the same way as the pipeline does (missing => np.nan)
row_df = make_single_row_frame(payload, feature_cols)
X = preprocessors.model_features.transform(row_df)

pred = severity_model.predict(X)[0]
print("Severity prediction:", pred)
if hasattr(severity_model, "predict_proba"):
    proba = severity_model.predict_proba(X)[0].tolist()
    print("Severity class probabilities:", proba)

Using severity run: 20260511T195003Z
Severity prediction: Minor
Severity class probabilities: [0.36619564535751997, 0.28018917097937135, 0.3536151836631086]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [19]:
# --- Demo (alternate): use the CLI predict command with JSON ---
# This uses the centralized loader to auto-pick the latest saved checkpoints.
import json
import subprocess
import sys

example = {
    "_comment": "Extra keys are ignored; missing keys are OK.",
    "Bridge_Type": "Beam",
    "Material": "Steel",
    "Age_Years": 25,
    "Failure_Type": "Scour",
}

out = subprocess.check_output([
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', CONFIG,
    '--project-root', PROJECT_ROOT,
    'predict',
    '--json', json.dumps(example),
], text=True)
print(out)

{
  "severity_pred": "Severe",
  "severity_proba": [
    0.3279340670475811,
    0.3096465625285445,
    0.36241937042387445
  ],
  "similar_cases": [
    {
      "Bridge_Name": "MetroSpan_95713",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 6.371624946594238
    },
    {
      "Bridge_Name": "RiverLink_43392",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 14.049088478088379
    },
    {
      "Bridge_Name": "SkyBridge_29471",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 15.08775520324707
    },
    {
      "Bridge_Name": "MetroSpan_53797",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 15.752196311950684
    },
    {
      "Bridge_Name": "GangaSetu_43718",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing